# Training on soft labels: typed-decisions

> **⚠️ Plan for a long run.** The full fine-tuning section trains every layer of the backbone on
> the full training data, and nothing can be cached when the backbone learns, so it takes far
> longer than head-only training: up to `BACKBONE_EPOCHS` passes through the backbone, forward and
> backward. ModernBERT-base is about a quarter of Qwen3-0.6B's size, so it runs several times
> faster than the decoder version of this notebook, but use a CUDA GPU (on Colab, an L4 or A100
> runtime). Save trained models to Google Drive with `model.save(...)`, since Colab deletes
> `/content` when the runtime disconnects. A run stopped with Ctrl-C keeps its best weights so far
> and continues from where it stopped when `fit` is called again with the same data.

[LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) has 1,200
training and 400 test records from four operational workflows: agent-trace monitoring, customer
service, invoice processing and security incidents. Each record is a structured JSON state with the
same five questions for its workflow.

Its labels are **soft**: each answer is a probability spread from several raters, for example
`{"continue": 0.28, "human_review": 0.43, "observe": 0.25, "stop": 0.03}`. sysone trains on these
distributions directly (passed as `gold`) and evaluates against them, so accuracy here means "picks
the raters' top answer", and **distance** shows how close the predicted spread is.

It trains ModernBERT-base, a bidirectional encoder, with `tuning="head"`, then fine-tunes the whole
backbone with `tuning="full"`. It runs faster than
[`decoder_typed_decisions.ipynb`](decoder_typed_decisions.ipynb), the same notebook with Qwen3-0.6B,
and the two can be compared directly.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.2"

In [ ]:
import json

from datasets import load_dataset

from sysone import EncoderModel, baselines, compare

BACKBONE_EPOCHS = 8  # upper bound for full fine-tuning; early stopping may end it sooner

## Load the data

`state`, `questions` and `gold` are JSON text, which sysone decodes itself. The workflow is used as
the row's `category`.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))
print("rater probabilities for one question:", json.loads(train[0]["gold"])["action"]["probabilities"])

## Before training: baselines

- **random guess**: every option equally likely.
- **average answer**: for each workflow question, the raters' average distribution in the
  training data. Every record in a workflow has the same questions, so this is a strong bar: a
  model that beats it is reading each record, not just learning the typical answer.

In [ ]:
results = baselines(test, train=train)
compare(results)

## Train

The soft `gold` distributions are the training targets. 10% of records are held back for
validation and early stopping, split evenly across workflows.

In [ ]:
model = EncoderModel(tuning="head")
model.fit(train)

## After training

In [ ]:
results["head"] = model.evaluate(test)
compare(results)

## Fine-tuning the whole backbone

With `tuning="full"`, every layer of ModernBERT trains along with the head. ModernBERT-base is small
enough for this to be affordable, and it is what works for this model: ModernBERT was pretrained only
to fill in masked words, and turning its `[CLS]` and `[MASK]` vectors into a decision means
reshaping the whole network. Small LoRA adapters barely moved it in our runs. (The decoder version
of this notebook uses LoRA and QLoRA, which suit Qwen.)

The backbone's learning rate is chosen for ModernBERT (`backbone_lr="auto"`, 5e-5). Validation runs
about ten times per epoch; training stops once validation loss hasn't improved for 3 epochs, and
the best weights are kept, so `BACKBONE_EPOCHS` is an upper bound.

In [ ]:
full = EncoderModel(tuning="full", epochs=BACKBONE_EPOCHS, batch_size=32)
full.fit(train)
results["full"] = full.evaluate(test)
compare(results)

**Metrics** (from `model.evaluate`, the same for hard and soft labels):
- **accuracy**: the model's most likely option is the right one (for soft labels: the raters'
  top answer). For a score question this means the exact level.
- **distance**: how far the predicted probabilities are from the label (0 = identical, 1 = completely
  different).
- **log_loss**: cross-entropy against the label; lower is better.
- **score_error**: for score questions, the gap between predicted and true expected level.
- **auc**: for yes/no questions, how well the model ranks yes above no (0.5 = chance).

With soft labels, **distance** and **log_loss** matter as much as accuracy: when raters split
45/55, either answer is defensible, but the probabilities should reflect the split.

In [ ]:
compare(results, by="overall")

### By workflow

In [ ]:
compare(results, by="category")

## Reading the results

- Raters often disagree, so accuracy has a ceiling well below 100%.
- **average answer** is the bar to beat. With a frozen backbone, the head has limited room to get
  past it; fine-tuning the whole backbone is what moves these numbers. In one run on a CUDA GPU
  with these settings, full fine-tuning reached about 0.79 test accuracy, against 0.49 for the
  average answer. Results vary a little between runs; change `seed` to see how much.
- Probabilities are not calibrated yet (temperatures stay at 1.0), which affects **distance** and
  **log_loss** more than accuracy.
- A long run stopped with Ctrl-C continues from where it stopped with `full.fit(train)`.